# Dinner-Table Training on Colab (no laptop training)

Trains all three learned models on a Colab GPU. Before running anything:

1. **Commit + push your work** — this notebook clones the repo, so anything
   uncommitted on your laptop will NOT be here. (`git status` must be clean.)
2. **Runtime → Change runtime type → T4 GPU** (free tier is fine for YOLO +
   VLM-LoRA; ACT full training needs a bigger GPU — see §5).
3. Fill `REPO_URL` and `HF_TOKEN` in the config cell below.

| Section | What | Rough cost (T4) |
| --- | --- | --- |
| §2 data slice | 120 teacher demos + YOLO labels + VQA + LeRobot export | ~1–2 h CPU |
| §3 YOLO | yolo11n detector, 50 epochs | ~20–40 min |
| §4 VLM LoRA | Qwen3-VL-2B + LoRA r16, 1 epoch slice | ~30–60 min |
| §5 ACT | smoke (50 steps) only | ~10 min |

Full-scale runs (3.4k demos, 100-ep YOLO, 3-ep SFT, 120k-step ACT) are sharded
procedures, not single sessions — each section says how.

In [ ]:
# 0 ─ GPU check + config ────────────────────────────────────────────────
!nvidia-smi -L

REPO_URL = "https://github.com/<org>/IntelAI.git"  # ← fill in
HF_TOKEN = ""  # ← huggingface.co/settings/tokens (write) for --push
SEED_BASE = 0   # shard id for demo_gen; use 0, 500, 1000, … across sessions

In [ ]:
# 1 ─ System libs (headless MuJoCo via EGL, same as the Dockerfile) ─────
!apt-get update -qq && apt-get install -y -qq libegl1 libgl1 ffmpeg >/dev/null 2>&1
!git clone {REPO_URL} dinner-table && cd dinner-table && git log --oneline -3

In [ ]:
# 2 ─ Python env. Plain pip: torch resolves to the CUDA build on Linux. ──
%cd /content/dinner-table
!pip install -q -e ".[cpu,dev]"
!python -c "import torch, mujoco; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available()); print('mujoco', mujoco.__version__)"

In [ ]:
# 3 ─ Smoke: one headless scene render proves MuJoCo+EGL work ───────────
!MUJOCO_GL=egl make smoke

## §2 — Data slice (teacher demos → labels → VQA → LeRobot)

The full dataset is 3,400 demos (days of CPU) — sharded with `--seed-base`
across sessions/machines and concatenated. This slice is enough to train and
validate every model end to end.

In [ ]:
# 4 ─ Teacher demos (CPU-bound; run in background, ~1–2 h for 120) ──────
!MUJOCO_GL=egl nohup python -m dinner_table.data.demo_gen --config dr_train --count 120 --seed-base {SEED_BASE} --out demos > demo_gen.log 2>&1 &
!sleep 60 && tail -5 demo_gen.log && ls demos | head

In [ ]:
# 5 ─ Labels + VQA + LeRobot export (run AFTER demo_gen finishes) ───────
!tail -3 demo_gen.log
!MUJOCO_GL=egl python -m dinner_table.data.labels datasets/yolo --demos demos --max-frames 1500
!python -m dinner_table.data.vqa_factory --demos demos --labels datasets/yolo --out datasets/vqa --train-n 1500 --val-n 200
!python -m dinner_table.data.lerobot_export --demos demos --root datasets/dinner
!ls datasets/yolo datasets/vqa datasets/dinner

## §3 — YOLO detector (`dinner-table/yolo`)

Smoke: 10 epochs to validate the loop, then the real 50-epoch slice run
(100 epochs on the full 6k-frame set for the final artifact).

In [ ]:
# 6 ─ YOLO smoke (10 epochs) ────────────────────────────────────────────
!python -m dinner_table.perception.yolo_train --data datasets/yolo --out artifacts/yolo_smoke.pt --epochs 10 --device 0

In [ ]:
# 7 ─ YOLO slice run + Hub upload ───────────────────────────────────────
!python -m dinner_table.perception.yolo_train --data datasets/yolo --out artifacts/yolo_best.pt --epochs 50 --device 0
!HF_TOKEN={HF_TOKEN} huggingface-cli upload dinner-table/yolo artifacts/yolo_best.pt

## §4 — VLM LoRA (`dinner-table/vlm-lora`)

Qwen3-VL-2B + LoRA r16. `--batch 1` fits a free T4; the final artifact uses
`--epochs 3 --batch 4` on a bigger GPU. `--push` uploads to the Hub repo.

In [ ]:
# 8 ─ VLM LoRA slice run (T4-safe) ──────────────────────────────────────
!HF_TOKEN={HF_TOKEN} python -m dinner_table.reasoning.sft_train --train-jsonl datasets/vqa/train.jsonl --val-jsonl datasets/vqa/val.jsonl --images datasets/vqa --out artifacts/vlm_lora --batch 1 --epochs 1 --push

## §5 — ACT policy (`dinner-table/act`)

**Free Colab runs the smoke only** (50 steps, no W&B) to validate the loop.
The full B9 run is ~120k steps over the complete `datasets/dinner` with
W&B logging — run it on your friend's RTX 4050 / Colab Pro / cloud GPU with:

```bash
WANDB_API_KEY=… python -m dinner_table.policies.studio_train --config act_dinner
```

(`studio_train` preflights the config, wraps `physicalai fit`, and records
provenance. The dataset can be pulled from the Hub instead of generated
locally once it is pushed.)

In [ ]:
# 9 ─ ACT smoke (validates config + data + physicalai fit wiring) ───────
!python -m dinner_table.policies.studio_train --config act_dinner --smoke

## §6 — Collect artifacts

Everything the i7/OpenVINO side needs: YOLO weights, VLM LoRA, ACT smoke
checkpoint, and the training logs.

In [ ]:
# 10 ─ Pack + back up ───────────────────────────────────────────────────
!ls -la artifacts/ && tar czf /content/training_artifacts.tgz artifacts/ demo_gen.log
from google.colab import drive, files
drive.mount('/content/drive')
!cp /content/training_artifacts.tgz "/content/drive/MyDrive/training_artifacts.tgz" && echo backed up